### Chunking

**Chunking** is the process of breaking a large document into **smaller pieces of text called chunks** so they can be individually embedded, stored, and retrieved.

For example:

```text
Large Document
      ↓
 ┌──────────┐
 │ Chunk 1  │
 ├──────────┤
 │ Chunk 2  │
 ├──────────┤
 │ Chunk 3  │
 ├──────────┤
 │ Chunk 4  │
 └──────────┘
```

In **RAG**, each chunk is converted into an **embedding** and stored in a vector database. When a query arrives, the system compares the query embedding with the chunk embeddings using **cosine similarity** and retrieves the most relevant chunks.

> **Chunking = breaking documents into smaller pieces so RAG can efficiently find and retrieve the relevant information.**


In [1]:
document = """
Dogs are loyal domestic animals and are commonly kept as pets.

Cats are independent domestic animals. They are also commonly kept
as household pets.

Elephants are intelligent animals, but they are not practical
household pets and require specialized environments.

A CPU executes instructions and performs calculations inside
a computer system.
"""

In [2]:
chunks = document.split("\n\n")

for i, chunk in enumerate(chunks):
    print(f"Chunk {i}:")
    print(chunk)
    print()

Chunk 0:

Dogs are loyal domestic animals and are commonly kept as pets.

Chunk 1:
Cats are independent domestic animals. They are also commonly kept
as household pets.

Chunk 2:
Elephants are intelligent animals, but they are not practical
household pets and require specialized environments.

Chunk 3:
A CPU executes instructions and performs calculations inside
a computer system.




Now the important part: **how big should a chunk be?**

```text
Too small
→ loses context

Too large
→ retrieval becomes less precise
→ more irrelevant text gets sent to the LLM
```

For example, imagine:

```text
Chunk A:
"Products must withstand temperatures between 0°C and 50°C."

Chunk B:
"Testing must be performed according to Section 7."
```

If we split too aggressively, we might retrieve B without the context that tells us **what is being tested**.

That's why chunking often uses **overlap**.

Conceptually:

```text
ABCDEFGHIJ
       HIJKLMNO
              MNOPQRSTUV
```

The chunks share some content.

The reason is simple:

> **Important information can cross a chunk boundary.**

Without overlap:

```text
... requirement A | requirement B ...
```

you could accidentally separate related information.

With overlap:

```text
... requirement A REQ | REQ requirement B ...
```

the model has a better chance of retaining the connection.

**But don't memorize `chunk_size = 10` or `overlap = 3`.** Those numbers are purely for demonstrating the mechanism.


In [3]:
text = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"

chunk_size = 10
overlap = 3

chunks = []

start = 0

while start < len(text):
    end = start + chunk_size
    chunks.append(text[start:end])
    start += chunk_size - overlap

for chunk in chunks:
    print(chunk)

ABCDEFGHIJ
HIJKLMNOPQ
OPQRSTUVWX
VWXYZ


In [4]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [8]:
query = "What comes after ABCDEFG?"

chunk_embeddings = model.encode(chunks)
query_embedding = model.encode(query)

In [9]:
from sklearn.metrics.pairwise import cosine_similarity

scores = cosine_similarity([query_embedding], chunk_embeddings)[0]

for i, score in enumerate(scores):
    print(f"Chunk {i}: {score}")

Chunk 0: 0.5974507331848145
Chunk 1: 0.2052069902420044
Chunk 2: 0.2672457695007324
Chunk 3: 0.14634397625923157
